# Testing the saved model
One cell code from ChatGPT.

In [22]:
import torch
import torch.nn as nn
import torchvision
from PIL import Image
import os

# ----------------------------
# 1️⃣ Device
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# 2️⃣ Recreate Model Architecture
# ----------------------------
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
model = torchvision.models.efficientnet_b0(weights=None).to(device)

# IMPORTANT: Must match training classes
class_names = ["ai", "real"]   # or manually define list
output_shape = 2

model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(1280, output_shape, bias=True)
).to(device)

# ----------------------------
# 3️⃣ Load Saved Weights
# ----------------------------
model.load_state_dict(torch.load("veriface_v1.pth", map_location=device))
model.eval()

# ----------------------------
# 4️⃣ Image Transforms (same as training)
# ----------------------------
auto_transforms = weights.transforms()

# ----------------------------
# 5️⃣ Prediction Function
# ----------------------------
def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = auto_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()

    return class_names[pred_class], probs[0][pred_class].item()

# ----------------------------
# 6️⃣ Use It
# ----------------------------
image_path = os.path.join("dataset","test_data_v2","577b964f98b34d839d7a88f22bf5c03c.jpg")
label, confidence = predict_image(image_path)

print(f"Prediction: {label}")
print(f"Confidence: {confidence:.4f}")

Prediction: ai
Confidence: 0.5047
